# Closed-unit pairing pilot — completed

Six pairs and three segmented whole-network variants were constructed and simulated on **03, 22 and 55**, using previously validated local collapsed sections and unchanged parent connectors. The prior frozen model, calibration, reference sections and response tables are intact.

**Scope:** 27 successful runs: 6 pairs, 3 segmented whole-network variants, 12 controlled single edits, and 6 parent/legacy-reference reruns. Five composition tests passed. All states settled under the existing 96-hour guard criterion. Parent and legacy-reference reruns reproduce the previous saved discharge traces to tolerance. This is a deliberately selected feasibility pilot, not a new generalization score.

The candidate audit found 435 eligible outer blocks and 420 adjacent pairs across 15 networks. Network 13 was initially considered, but 18 of its 30 saved section geometries have endpoint offsets of approximately 7–14 m relative to parent anchors. It was excluded from this first composition pilot; no snapping or hydraulic alteration was applied. All selected blocks in 03/22/55 have matching coordinates at both anchors.

The new segmented representation replaces each outer unit separately; it preserves connectors and tails, and has section-specific length, width, depth, slope and frozen routing parameters. It is not the old uniform line with extra nodes. No new hydraulic calibration or raster skeletonization was performed.


Detailed endpoint and length checks are saved in `geometric_anchor_audit.csv` and `geometry_length_audit.csv`; `audit_geometry.py` reproduces them.

In [1]:
from pathlib import Path
import sys,json
import pandas as pd
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'RAPID/src').exists())
sys.path.insert(0,str(ROOT/'RAPID/investigations/closed_unit_pairing_20261019'))
from review_pilot import pair_map,hydrograph
OUT=ROOT/'RAPID/outputs/hydrograph_closed_unit_pairing_20261019'
selection=pd.read_csv(OUT/'selected_pairs.csv',dtype={'example':str})
responses=pd.read_parquet(OUT/'responses.parquet')
linked=pd.read_parquet(OUT/'linked_diagnostics.parquet')
primary=linked[linked.window.eq('0.5%')].copy()
display(selection[['example','state_a','state_b','pair_type','parent_mean_length_m','connector_fraction']])

,example,state_a,state_b,pair_type,parent_mean_length_m,connector_fraction
0,03,S002_unit_1,S003_unit_4,simple+simple,2423.539149,0.323994
1,03,CB001_bubble_14,S030_unit_30,compound+compound,7424.392814,0.260185
2,55,S071_unit_9,S065_unit_6,simple+simple,7742.036448,0.661256
3,55,S107_unit_51,CB023_bubble_23,compound+simple,8018.460317,0.454555
4,22,CB005_bubble_13,S180_unit_331,compound+simple,24300.444947,0.465933
5,22,CB003_bubble_10,CB002_bubble_11,compound+compound,135763.990120,0.078539


## Inspect a pair

Choose a row below. Maps are zoomed to the enclosing domain, with unit A, unit B and the retained connector distinguished. Hydrographs compare the original parent, controlled A-only/B-only edits and the combined edit. Vertical dotted lines mark excess-volume centroids. The controls are newly composed from the same original parent; they do not assume that unrelated geometry in every historical child was unchanged.


In [7]:
PAIR_INDEX=0  # 0..5; index 5 is the large compound+compound pair in 22
pair=selection.iloc[PAIR_INDEX]
display(primary[(primary.example==pair.example)&primary.state.eq(pair.state)][['example','state','centroid_shift_hours','peak_shift_hours','peak_excess_change_pct','duration80_change_pct','volume_change_pct','asymmetry_change']].T)
pair_map(pair.example,pair.state).show()
hydrograph(pair.example,pair.state).show()

,1
example,03
state,pair_S002_unit_1__S003_unit_4
centroid_shift_hours,-0.123147
peak_shift_hours,-0.05
peak_excess_change_pct,0.420288
duration80_change_pct,-0.066284
volume_change_pct,0.12558
asymmetry_change,0.001047


## Segmented versus legacy uniform centerline

The revised child is a series of local collapsed units plus original connectors/tails. It has 15 collapsed blocks in 03, 14 in 22 and 56 in 55. Section width variability is retained; child route entropy and route-length SD are zero because there is only one path. Legacy and new states are compared under the same pilot window and baseline.

The primary centroid advances (legacy → segmented) are approximately **2.90 → 2.18 h in 03**, **11.58 → 5.26 h in 22**, and **18.57 → 17.28 h in 55**. The difference is not a connector-only causal effect: the collapsed unit sections also differ from the globally uniform reference in width, depth and slope.


In [3]:
EXAMPLE='22'  # 03, 22, 55
hydrograph(EXAMPLE).show()
display(primary[primary.comparison.isin(['segmented_whole_network','legacy_uniform_whole_network'])][['example','comparison','centroid_shift_hours','peak_shift_hours','peak_excess_change_pct','duration80_change_pct','volume_change_pct','asymmetry_change']])

,example,comparison,centroid_shift_hours,peak_shift_hours,peak_excess_change_pct,duration80_change_pct,volume_change_pct,asymmetry_change
0,03,legacy_uniform_whole_network,-2.897389,-1.166667,9.411139,0.555468,2.037082,0.037970
3,03,segmented_whole_network,-2.179386,-2.183333,4.371515,1.210966,1.517676,0.023830
24,22,legacy_uniform_whole_network,-11.583453,-10.066667,17.911856,-20.533046,-3.837578,-0.006882
27,22,segmented_whole_network,-5.263388,-5.716667,4.655735,-7.299252,-1.786033,-0.000130
48,55,legacy_uniform_whole_network,-18.573308,-18.400000,3.139437,-4.466435,-1.596918,0.008590
51,55,segmented_whole_network,-17.279029,-17.166667,2.541939,-3.997642,-1.488752,0.008482


## Enclosing length versus edited length

Existing seven-input predictions use the enclosing domain exactly as before. Additional descriptors report

$$\mu_{L,p}^{\mathrm{edit}}=\sum_{e\in E_{\mathrm{edit},p}}r_eL_e,$$

with probabilities computed on the connected enclosing graph, and the corresponding child value and ratio. Connectors remain in the graph and simulation. The unchanged fraction is the complementary fraction of probability-weighted mean length; for the whole-network variant it includes tails as well as connectors.

This pilot does **not** replace the existing size variable or refit the model with edited-region summaries. Those would require a new, consistently constructed feature table for the original training comparisons too.

The large pair in 22 covers 135.8 km of parent mean travelled distance, of which 125.1 km is edited. Other selected pairs have unchanged fractions of 26–66%; therefore enclosing size and edited size can differ substantially even for valid adjacent units.


In [4]:
cols=['example','state','kind','B_parent_route_length_mean','edited_parent_mean_length_m','edited_child_mean_length_m','edited_mean_length_ratio','unchanged_mean_length_m','unchanged_length_fraction','B_ratio_width_mean','B_child_width_cv']
display(primary[primary.comparison.isin(['pair','segmented_whole_network'])][cols])

,example,state,kind,B_parent_route_length_mean,edited_parent_mean_length_m,edited_child_mean_length_m,edited_mean_length_ratio,unchanged_mean_length_m,unchanged_length_fraction,B_ratio_width_mean,B_child_width_cv
1,03,pair_S002_unit_1__S003_unit_4,pair,2423.539149,1638.327684,1493.160777,0.911393,785.211465,0.323994,1.844320,0.057198
2,03,pair_CB001_bubble_14__S030_unit_30,pair,7424.392814,5492.677361,4829.241292,0.879214,1931.715454,0.260185,2.590384,0.205948
3,03,all_closed_units_collapsed,segmented_whole_network,61739.974601,28747.655743,26351.151827,0.916637,32992.318858,0.534375,1.686956,0.205580
25,22,pair_CB005_bubble_13__S180_unit_331,pair,24300.444947,12978.077177,11657.978500,0.898282,11322.367770,0.465933,1.743790,0.304294
26,22,pair_CB003_bubble_10__CB002_bubble_11,pair,135763.990120,125101.285983,114464.803566,0.914977,10662.704137,0.078539,4.680524,0.175118
27,22,all_closed_units_collapsed,segmented_whole_network,384676.634132,290206.049716,262995.810425,0.906238,94470.584416,0.245584,3.840797,0.259991
49,55,pair_S071_unit_9__S065_unit_6,pair,7742.036448,2622.569816,2424.501439,0.924475,5119.466633,0.661256,1.324265,0.029613
50,55,pair_S107_unit_51__CB023_bubble_23,pair,8018.460317,4373.625262,4132.272652,0.944816,3644.835055,0.454555,1.619774,0.052466
51,55,all_closed_units_collapsed,segmented_whole_network,268872.418153,198509.800136,187818.751689,0.946143,70362.618017,0.261695,2.070753,0.150988


## Combined responses and prediction diagnostics

Centroid shifts in these six serial-edit cases are nearly additive: combined minus A-only minus B-only differs by less than 8 seconds in the primary window. This is consistent with the fixed linear routing setup and does not imply nonlinear/general hydrological responses are additive. The finite-window response definition still matters. Other responses, including global peak time, need not be additive.

New cases are correlated interventions within existing networks. They expand geometric/edit coverage, not the count of independent network examples.

The prediction table uses a reconstruction of the original seven-input forest trained on the other 14 networks; original held-network predictions were reproduced before predicting pilot states. No new cases enter fitting. Large segmented-network shifts remain severely underpredicted. A training-augmentation test has **not** yet been run. Do not interpret six deliberately selected pairs as an aggregate accuracy benchmark.


In [5]:
a=pd.read_csv(OUT/'additivity_diagnostics.csv',dtype={'example':str})
a=a[(a.window=='0.5%')&(a.response=='centroid_shift_hours')].copy()
a['nonadditivity_seconds']=a.nonadditivity*3600
display(a[['example','state','combined','single_a','single_b','nonadditivity_seconds']])
display(primary[primary.comparison.isin(['pair','segmented_whole_network'])][['example','state','centroid_shift_hours','predicted_hours','centroid_error_minutes']])

,example,state,combined,single_a,single_b,nonadditivity_seconds
0,03,pair_S002_unit_1__S003_unit_4,-0.123147,-0.052531,-0.070504,-0.402916
18,03,pair_CB001_bubble_14__S030_unit_30,-0.540705,-0.376087,-0.164758,0.503314
36,55,pair_S071_unit_9__S065_unit_6,-0.210459,-0.100012,-0.110428,-0.066936
54,55,pair_S107_unit_51__CB023_bubble_23,-0.323107,-0.160963,-0.162103,-0.149870
72,22,pair_CB005_bubble_13__S180_unit_331,-0.275016,-0.039863,-0.235122,-0.109381
90,22,pair_CB003_bubble_10__CB002_bubble_11,-1.991129,-0.599637,-1.389447,-7.363118


,example,state,centroid_shift_hours,predicted_hours,centroid_error_minutes
1,03,pair_S002_unit_1__S003_unit_4,-0.123147,-0.133204,-0.603390
2,03,pair_CB001_bubble_14__S030_unit_30,-0.540705,-0.654178,-6.808375
3,03,all_closed_units_collapsed,-2.179386,-0.951499,73.673248
25,22,pair_CB005_bubble_13__S180_unit_331,-0.275016,-1.218340,-56.599446
26,22,pair_CB003_bubble_10__CB002_bubble_11,-1.991129,-1.527022,27.846414
27,22,all_closed_units_collapsed,-5.263388,-1.513681,224.982463
49,55,pair_S071_unit_9__S065_unit_6,-0.210459,-0.135284,4.510529
50,55,pair_S107_unit_51__CB023_bubble_23,-0.323107,-0.386980,-3.832394
51,55,all_closed_units_collapsed,-17.279029,-1.030514,974.910902


## Common-window and numerical checks

The new fast responses start earlier than the previous shared-window onset in some cases. We therefore created a separate pilot response version using the union of the historical boundaries and new-state boundaries. **Every pilot state, including parent and legacy reference, was recomputed on that common window.** The original baseline is unchanged and all original tables remain intact. At 0.5%, onset moves earlier by 24 minutes (03), 82 minutes (22) and 1 minute (55); no endpoint extension was needed. Separate 1% and 2% results are available. These recalculated centroid numbers must not be silently pooled with historical-window metrics.

Network 22 retains its pre-existing provisional source-clock qualification. Geometry-only new predictors contain no routed depth/celerity/response-derived information; depth and slope remain routing inputs and audit fields.


In [6]:
w=pd.read_csv(OUT/'window_audit.csv',dtype={'example':str})
w['earlier_onset_minutes']=(w.previous_rise_seconds-w.rise_seconds)/60
w['later_end_minutes']=(w.fall_seconds-w.previous_fall_seconds)/60
display(w[['example','window','expanded','earlier_onset_minutes','later_end_minutes','baseline_cms','previous_baseline_cms']])
display(pd.read_csv(OUT/'reference_reproduction.csv',dtype={'example':str}))
display(responses[(responses.comparison.isin(['pair','segmented_whole_network']))][['example','state','window','centroid_shift_hours','peak_shift_hours']])

,example,window,expanded,earlier_onset_minutes,later_end_minutes,baseline_cms,previous_baseline_cms
0,03,0.5%,True,24.0,0.0,1520.700000,1520.700000
1,03,1%,True,39.0,0.0,1520.700000,1520.700000
2,03,2%,True,55.0,0.0,1520.700000,1520.700000
3,22,0.5%,True,82.0,0.0,18825.528487,18825.528487
4,22,1%,True,95.0,0.0,18825.528487,18825.528487
5,22,2%,True,27.0,0.0,18825.528487,18825.528487
6,55,0.5%,True,1.0,0.0,231.394832,231.394832
7,55,1%,False,0.0,0.0,231.394832,231.394832
8,55,2%,False,0.0,0.0,231.394832,231.394832


,example,state,max_absolute_discharge_difference
0,03,S000_base,0.0
1,03,single_centerline,0.0
2,22,S000_base,0.0
3,22,single_centerline,0.0
4,55,S000_base,0.0
5,55,single_centerline,0.0


,example,state,window,centroid_shift_hours,peak_shift_hours
1,03,pair_S002_unit_1__S003_unit_4,0.5%,-0.123147,-0.050000
2,03,pair_CB001_bubble_14__S030_unit_30,0.5%,-0.540705,-0.783333
3,03,all_closed_units_collapsed,0.5%,-2.179386,-2.183333
9,03,pair_S002_unit_1__S003_unit_4,1%,-0.116060,-0.050000
10,03,pair_CB001_bubble_14__S030_unit_30,1%,-0.546593,-0.783333
11,03,all_closed_units_collapsed,1%,-2.170899,-2.183333
17,03,pair_S002_unit_1__S003_unit_4,2%,-0.105954,-0.050000
18,03,pair_CB001_bubble_14__S030_unit_30,2%,-0.545808,-0.783333
19,03,all_closed_units_collapsed,2%,-2.126811,-2.183333
25,22,pair_CB005_bubble_13__S180_unit_331,0.5%,-0.275016,-0.466667


## Artifacts and next decision

- `candidate_pairs.csv`, `block_audit.csv`, `selection.json`: candidate inventory and geometry-only selection.
- `variants/<example>/<state>/`: section-specific parameters, actual GPKG geometry and parent–child lineage.
- `section_hydraulic_audit.csv`: reused single-section physical properties and boundary validation.
- `runs/`: separate completed RAPID simulations, receipts and forcing.
- `responses.parquet`, `state_metrics.parquet`, `window_audit.csv`: new pilot response version and sensitivities.
- `predictors.parquet`, `linked_diagnostics.parquet`, `predictor_support.csv`: conventional and edited-region descriptors plus held-network predictions.
- `figures/`: standalone interactive maps and hydrographs.
- `preparation_manifest.json`, `routing_plan.json`, `analysis_manifest.json`: provenance and scope.

Scripts are in `RAPID/investigations/closed_unit_pairing_20261019/`: `pilot.py --prepare`, `run_pilot.py --run`, `analyze_pilot.py`, `review_pilot.py`, and `test_pilot.py`. Preparation/routing write only the dedicated pilot directory. Preserve versioned outputs if code or geometry changes; do not reuse old run receipts for new parameter hashes.

**Recommendation:** review these six maps and the three whole-network comparisons first. The foundational construction works. Decide next whether to expand combinations and/or test edited-region size as a distinct feature representation. Retain the existing model until a grouped augmentation experiment demonstrates an improvement; keep whole-network and pair scores separate from original partial-collapse scores.


## Why the two whole-network representations differ in 22

Completed focused hydraulic and refinement audit: `segmented_22_audit/`, reproduced by `inspect_segmented_22.py`. This extends the pilot; it does not modify any fitted model or original response table.

| Quantity | Uniform reference | Segmented single path |
|---|---:|---:|
| Total geometric length | 357.328 km | 357.466 km |
| Total head drop | 78.544 m | 78.544 m |
| Celerity | 5 m/s everywhere (capped from raw 7.820 m/s) | 1.163–5 m/s |
| Sum of section L/c | 19.852 h | 27.491 h |
| Pilot finite-window centroid advance | 11.584 h | 5.263 h |

Manning n (0.0147211), diffusion multiplier (0.5), total head drop and the original native time step (4.2857 s) are identical. The length difference alone corresponds to about 28 seconds at 5 m/s. Differences arise primarily from retaining the spatial slope profile and section depths: the uniform reference assigns the mean slope everywhere and reaches the 5 m/s cap throughout.

Using segmented lengths but global slope/depth gives 19.859 h of L/c. Keeping the global depth while substituting local slopes increases this to 25.448 h. Using local depths with the global slope gives 19.866 h; both local fields give 27.491 h. These are formula-based kinematic controls, not new physical simulations or unique causal allocations. Width affects assigned depth under the existing hydraulic policy, and both width and slope also affect spreading.

Relative to 5 m/s over the same length, unchanged connectors/tails contribute 4.648 h of additional L/c, and collapsed-unit sections contribute 2.984 h. The largest single contributor is unchanged parent reach 27: length 16.283 km, width 1723.6 m, depth 8.63 m, head drop only 0.1871 m (slope ≈1.15×10⁻⁵), celerity 1.615 m/s. It adds 1.897 h relative to 5 m/s—about 25% of the additional travel time. Other slow connectors and local sections account for the remaining 75%. This is distributed slow routing, not evidence of a newly introduced blockage or simulated backwater mechanism.

**Numerical check:** reran both states at half the time step, twice the cells and four times the cells (with half time step). All remained stable/settled and nonnegative. The uniform refined output minimum is ~0.548 m³/s below the pilot baseline, so this audit uses a new shared minimum across refinements and recomputes the parent centroid consistently, keeping the pilot window fixed. On that common diagnostic baseline the centroid gap is ~6.311 h and varies by less than one second across refinements. Thus discretization does not explain the hours-scale centroid difference. Global peak timing is less robust: refinement moves the uniform peak by up to 20 minutes and the segmented peak by up to 3 minutes. Keep these separate from the stable centroid conclusion.

Sum L/c is a routing-time diagnostic; it is not identical to the finite-window excess centroid. The latter also depends on spreading, baseline and window endpoints. The difference is internally consistent with the assigned hydraulics, but its real-world plausibility depends on the inherited slope/depth profile and celerity cap. The most useful physical follow-up is checking the very small head drops around reach 27 and adjacent sections against the source profile. No physical validation of those head differences is claimed here.
